# Topic: Encryption/Decryption Toolkit - AES-GCM authenticated encryption, PBKDF2 key derivation, and file stream locking
 
## 1. SYMMETRIC ENCRYPTION: AES-GCM
- **Symmetric Encryption**: Uses the same secret key for both encryption and decryption.
- **AES-GCM (Galois/Counter Mode)**:
  - An **Authenticated Encryption with Associated Data (AEAD)** cipher mode.
  - Provides both **Confidentiality** (hiding plaintext data) and **Integrity** (generating a cryptographic tag that guarantees the data has not been modified or tampered with).
- **Parameters**:
  1. **Initialization Vector (IV) / Nonce**: A unique, random 12-byte value that must NEVER be reused with the same key.
  2. **Tag**: An authentication tag (usually 16 bytes) generated during encryption, verified during decryption.
 
## 2. PASS-BASED KEY DERIVATION (PBKDF2)
- To securely convert a variable-length user password into a fixed 256-bit cryptographic AES key, we use **PBKDF2** (Password-Based Key Derivation Function 2) with a random Salt. This prevents precomputed brute-force dictionary lookups.
 
---


In [ ]:
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes

class FileLockManager:
    """Manages secure file encryption and decryption using AES-GCM and PBKDF2."""
    def __init__(self, password):
        self.password = password.encode('utf-8')

    def _derive_key(self, salt):
        """Derives a 256-bit AES key from password using PBKDF2HMAC."""
        kdf = PBKDF2HMAC(
            algorithm=hashes.SHA256(),
            length=32,  # 32 bytes = 256 bits key
            salt=salt,
            iterations=100000
        )
        return kdf.derive(self.password)

    def encrypt_file(self, source_path, dest_path):
        """Encrypts file. Writes: [Salt (16b)] + [IV (12b)] + [Tag+Ciphertext]."""
        # Read raw data
        with open(source_path, "rb") as f:
            plaintext = f.read()
            
        # Generate random parameters
        salt = os.urandom(16)
        iv = os.urandom(12)
        
        # Derive key and instantiate AESGCM
        key = self._derive_key(salt)
        aesgcm = AESGCM(key)
        
        # Encrypt (GCM mode automatically appends the auth tag to ciphertext)
        ciphertext = aesgcm.encrypt(iv, plaintext, None)
        
        # Write packed format to destination file
        with open(dest_path, "wb") as f:
            f.write(salt)
            f.write(iv)
            f.write(ciphertext)
        print(f"File encrypted successfully: {dest_path}")

    def decrypt_file(self, source_path, dest_path):
        """Decrypts file, verifying authentication tag integrity."""
        # Read packed binary payload
        with open(source_path, "rb") as f:
            salt = f.read(16)
            iv = f.read(12)
            ciphertext = f.read()
            
        # Derive key using extracted salt
        key = self._derive_key(salt)
        aesgcm = AESGCM(key)
        
        try:
            # Decrypt and verify integrity tag automatically
            plaintext = aesgcm.decrypt(iv, ciphertext, None)
            
            with open(dest_path, "wb") as f:
                f.write(plaintext)
            print(f"File decrypted successfully: {dest_path}")
            return True
        except Exception as e:
            # If tag verification fails (indicating tampered data or wrong password)
            print(f"[!] DECRYPTION ERROR: Authentication tag mismatch! Data corrupted or wrong password. Details: {e}")
            return False



In [ ]:
# Testing the Toolkit sandbox
print("--- Initializing AES-GCM Encryptor ---")
manager = FileLockManager("MySecurePassphrase123")

plain_file = "lab_secrets.txt"
enc_file = "lab_secrets.enc"
dec_file = "lab_secrets_restored.txt"

# Create sensitive secret file
with open(plain_file, "w", encoding="utf-8") as f:
    f.write("CONFIDENTIAL: Enterprise API credentials are admin/admin123.")

# 1. Run Encryption
manager.encrypt_file(plain_file, enc_file)

# 2. Run Decryption
manager.decrypt_file(enc_file, dec_file)

# Verify restored file matches
with open(dec_file, "r", encoding="utf-8") as f:
    print(f"Restored Content: {repr(f.read())}")



In [ ]:
# 3. Simulating Tampering Attack
print("\n--- Simulating Tampering Attack on Encrypted File ---")
with open(enc_file, "r+b") as f:
    f.seek(50)  # Seek arbitrary offset
    f.write(b"\xff")  # Corrupt 1 byte in the ciphertext payload

# Try to decrypt corrupted file
manager.decrypt_file(enc_file, "corrupt_output.txt")
# Expected: Decryption fails with InvalidTag / Exception!



In [ ]:
# Clean up files
for file in [plain_file, enc_file, dec_file, "corrupt_output.txt"]:
    if os.path.exists(file):
        os.remove(file)
